# E-commerce Sales Data Cleaning Project

This notebook documents the complete process of cleaning a messy e-commerce sales dataset using Python and Pandas.

The dataset contains several common real-world data issues, including:

- non-numeric values in numeric fields
- negative values
- inconsistent or invalid dates
- incorrect total calculations
- inconsistent category names
- mixed casing in text fields
- extra whitespace
- currency symbols in price fields

The goal is to transform the dataset into a clean, consistent, and analysis-ready format.


## Importing required libraries

We start by importing Pandas, which will be used for all data cleaning steps.


In [1]:
import pandas as pd


## Loading the raw dataset

We load the original CSV file and preview the first few rows to understand the structure and identify potential issues.


In [2]:
df = pd.read_csv("messy_ecommerce_sales_data.csv")
df.head()


,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total
0,100,Customer_100,ORD-41285,11/22/2024,Blender,Home,3,38,Cash on Delivery,Shipped,114.00
1,101,Customer_101,ORD-35783,7/5/2025,Smartphone,Electronics,2,abd,PayPal,Processing,NaN
2,102,Customer_102,ORD-84355,12/23/2024,Tennis Racket,Sports,1,389.05,PayPal,Delivered,389.05
3,103,Customer_103,ORD-57811,3/19/2025,Science,Books,5,233.92,PayPal,Processing,1169.60
4,104,Customer_104,ORD-93614,10/20/2025,Biography,Books,1,552.51,Cash on Delivery,Processing,552.51


## Initial inspection

Before cleaning, we check:

- column names
- data types
- missing values
- general structure

This helps us determine which columns require cleaning and what transformations are needed.


In [3]:
df.info()
df.isna().sum()
df.columns


<class 'pandas.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ID              103 non-null    int64  
 1    Customer_Name  103 non-null    str    
 2   Order_ID        103 non-null    str    
 3   Order_Date      103 non-null    str    
 4   Product         103 non-null    str    
 5    Category       95 non-null     str    
 6   Quantity        98 non-null     str    
 7   Price           98 non-null     str    
 8   Payment_Method  103 non-null    str    
 9   Status          103 non-null    str    
 10  Total           89 non-null     float64
dtypes: float64(1), int64(1), str(9)
memory usage: 9.0 KB


Index(['ID', ' Customer_Name', 'Order_ID', 'Order_Date', 'Product',
       ' Category', 'Quantity', 'Price', 'Payment_Method', 'Status', 'Total'],
      dtype='str')

## Cleaning column names

Some column names contain leading or trailing spaces.
We remove unnecessary whitespace to ensure consistent column naming.


In [4]:
df.columns = df.columns.str.strip()
df.columns


Index(['ID', 'Customer_Name', 'Order_ID', 'Order_Date', 'Product', 'Category',
       'Quantity', 'Price', 'Payment_Method', 'Status', 'Total'],
      dtype='str')

## Cleaning the Quantity column

The Quantity column contains non-numeric values and negative values.

Steps performed:

1. Convert all values to numeric (invalid values become NaN)
2. Remove negative and NaN values
3. Replace the original column with the cleaned version
4. Reset the index


In [5]:
df["Quantity_Numeric"] = pd.to_numeric(df["Quantity"], errors="coerce")
df = df[df["Quantity_Numeric"] >= 0]
df = df.drop(columns=["Quantity"])
df = df.rename(columns={"Quantity_Numeric": "Quantity"})
df = df.reset_index(drop=True)


## Cleaning the Price column

The Price column contains currency symbols, whitespace, and non-numeric values.

Steps performed:

1. Remove currency symbols
2. Strip whitespace
3. Convert to numeric
4. Remove invalid or negative values
5. Replace the original Price column


In [6]:
df["Price_Clean"] = (
    df["Price"]
    .astype(str)
    .str.strip()
    .str.replace("$", "", regex=False)
    .str.replace("€", "", regex=False)
    .str.replace("kr", "", regex=False)
)

df["Price_Numeric"] = pd.to_numeric(df["Price_Clean"], errors="coerce")
df = df[df["Price_Numeric"] >= 0]
df = df.drop(columns=["Price", "Price_Clean"])
df = df.rename(columns={"Price_Numeric": "Price"})
df = df.reset_index(drop=True)


## Recalculating the Total column

The Total column contains incorrect values.
To ensure accuracy, we recalculate it using:

Total = Quantity * Price


In [7]:
df["Total"] = df["Quantity"] * df["Price"]
df["Total"].head()


0     114.00
1     389.05
2    1169.60
3     552.51
4     366.18
Name: Total, dtype: float64

## Cleaning the Order_Date column

The Order_Date column contains inconsistent formats and one invalid entry.

Steps performed:

1. Convert all values to datetime
2. Identify invalid dates
3. Correct the one problematic entry
4. Re-parse the column to ensure consistency


In [8]:
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")
df.loc[df["Order_ID"] == "ORD-77417", "Order_Date"] = "2023-01-05"
df["Order_Date"] = pd.to_datetime(df["Order_Date"])


## Cleaning Category and Product columns

These columns contain inconsistent casing, misspellings, and extra whitespace.

Steps performed:

- strip whitespace
- convert to lowercase
- fix known misspellings
- standardize formatting
- convert to title case (Product) or capitalized (Category)


In [9]:
df["Category"] = (
    df["Category"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({"electronic": "electronics", "electonics": "electronics"})
    .str.capitalize()
    .fillna("Unknown")
)

df["Product"] = (
    df["Product"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({"shoes": "Shoes"})
    .str.title()
)


## Cleaning remaining text columns

The following columns contain unnecessary whitespace:

- Customer_Name
- Payment_Method
- Status
- Order_ID

We remove leading and trailing spaces to ensure consistency.


In [10]:
df["Customer_Name"] = df["Customer_Name"].str.strip()
df["Payment_Method"] = df["Payment_Method"].str.strip()
df["Status"] = df["Status"].str.strip()
df["Order_ID"] = df["Order_ID"].str.strip()


## Final dataset validation

We perform a final check to confirm that:

- no missing values remain
- all data types are correct
- all columns are clean and standardized
- the dataset is ready for analysis or export


In [11]:
df.info()
df.isna().sum()
df.head()
df.tail()


<class 'pandas.DataFrame'>
RangeIndex: 87 entries, 0 to 86
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   ID              87 non-null     int64         
 1   Customer_Name   87 non-null     str           
 2   Order_ID        87 non-null     str           
 3   Order_Date      87 non-null     datetime64[us]
 4   Product         87 non-null     str           
 5   Category        87 non-null     str           
 6   Payment_Method  87 non-null     str           
 7   Status          87 non-null     str           
 8   Total           87 non-null     float64       
 9   Quantity        87 non-null     float64       
 10  Price           87 non-null     float64       
dtypes: datetime64[us](1), float64(3), int64(1), str(6)
memory usage: 7.6 KB


,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Payment_Method,Status,Total,Quantity,Price
82,198,Customer_198,ORD-14608,2025-07-27,Vacuum,Unknown,Cash on Delivery,Shipped,994.02,2.0,497.01
83,199,Customer_199,ORD-82922,2025-01-22,Blender,Home,Credit Card,Shipped,1861.40,5.0,372.28
84,175,Customer_175,ORD-56651,2025-02-24,Headphones,Electronics,Credit Card,Processing,111.36,1.0,111.36
85,142,Customer_142,ORD-69018,2025-10-30,Shoes,Clothing,Credit Card,Shipped,3226.30,5.0,645.26
86,146,Customer_146,ORD-32755,2025-07-09,Basketball,Electronics,Bank Transfer,Processing,1410.84,2.0,705.42


## Saving the cleaned dataset

We export the cleaned dataset to a new CSV file named:

cleaned_ecommerce_sales_data.csv


In [12]:
df.to_csv("cleaned_ecommerce_sales_data.csv", index=False)
